In [2]:
"""
Step 1 — Missing / Placeholder Data Handling + Deduplication

Input  : Movies_Reviews_modified_version1.csv
Output : movies_reviews_step1_cleaned.csv
Chart  : step1_missing_data_chart.png
"""

# -- Standard library ----------------------------------------------------------
import ast                       # safe string-to-list parsing
from collections import Counter  # used to find the mode genre list

# -- Third-party ---------------------------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

# -- Font for nicer charts (falls back silently if unavailable) ----------------
plt.rcParams["font.family"] = "DejaVu Sans"

INPUT_FILE  = "Movies_Reviews_modified_version1.csv"
OUTPUT_FILE = "movies_reviews_step1_cleaned.csv"
CHART_FILE  = "step1_missing_data_chart.png"

UNKNOWN_MOVIE_PLACEHOLDER = "Unknown"        # sentinel value used in movie_name
UNKNOWN_GENRE_FALLBACK    = ["Unknown genre"]  # imputed when no genre exists

def parse_genres(cell) -> list:
    """
    Safely convert a genres cell from its CSV string form to a Python list.

    The CSV stores genres like: "['Comedy', 'Drama']"  (a string, not a list).
    ast.literal_eval turns that string into an actual Python list.

    Edge cases handled:
      - NaN / None  -> returns []
      - Already a list (e.g. after in-memory re-use) -> returned as-is
      - Malformed string that cannot be parsed -> returns []
    """
    if pd.isna(cell):
        return []
    if isinstance(cell, list):
        return cell
    try:
        parsed = ast.literal_eval(str(cell))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        # Cell contains something that looks like text but is not a list literal
        return []

def most_common_genre_list(genre_series: pd.Series):
    """
    Given a pandas Series of genre lists belonging to one movie, return the
    single most frequent NON-EMPTY genre list (tuple -> back to list).

    Returns None if every entry in the series is an empty list
    (meaning no genre data exists anywhere for this movie).

    Example
    -------
    Input  : [['Action'], [], ['Action'], ['Comedy']]
    Non-empty tuples: [('Action',), ('Action',), ('Comedy',)]
    Counter most_common: ('Action',) with count 2
    Output : ['Action']
    """
    non_empty = [tuple(g) for g in genre_series if len(g) > 0]
    if not non_empty:
        return None  # flag: this movie has zero genre info
    mode_tuple, _ = Counter(non_empty).most_common(1)[0]
    return list(mode_tuple)

print("=" * 70)
print("STEP 1: Missing / Placeholder Data Handling + Deduplication")
print("=" * 70)

df = pd.read_csv(INPUT_FILE)
print(f"\n[Load]  Rows: {len(df):,}  |  Columns: {list(df.columns)}")

df["genres"] = df["genres"].apply(parse_genres)
print(f"\n[Parse] `genres` column converted from strings to Python lists.")

# Snapshot before any modifications — used in the EDA chart.
before_unknown_movie  = (df["movie_name"] == UNKNOWN_MOVIE_PLACEHOLDER).sum()
before_empty_genres   = df["genres"].apply(len).eq(0).sum()
before_dedup_total    = len(df)
before_dup_reviews    = df["Reviews"].duplicated(keep=False).sum()

print(f"\n[Before] 'Unknown' movie_name rows : {before_unknown_movie:,}")
print(f"[Before] Empty genres [] rows       : {before_empty_genres:,}")
print(f"[Before] Total rows                 : {before_dedup_total:,}")
print(f"[Before] Rows with duplicate Reviews: {before_dup_reviews:,}")

# Flag but do NOT drop — boolean column lets downstream members decide what to do.
df["is_unknown_movie_name"] = df["movie_name"] == UNKNOWN_MOVIE_PLACEHOLDER

n_flagged = df["is_unknown_movie_name"].sum()
print(f"\n[Flag]  {n_flagged:,} rows flagged as 'Unknown' movie_name "
      f"(column: is_unknown_movie_name).")

# Per-movie mode genre handles two problems in one pass:
#   a) fills empty-genre rows from sibling rows of the same movie
#   b) resolves conflicting genre lists across rows for the same movie
movie_mode_genres = (
    df.groupby("movie_name")["genres"]
    .apply(most_common_genre_list)
    .to_dict()
)

n_conflict_movies = sum(
    1
    for movie, group in df.groupby("movie_name")
    if len({tuple(g) for g in group["genres"] if len(g) > 0}) > 1
)
print(f"\n[Genre] Movies with conflicting genre lists (before fix): "
      f"{n_conflict_movies:,}")

def apply_movie_mode_genre(row):
    """
    Replace the row's genre with the movie's canonical (mode) genre.
    If the movie has NO non-empty genre anywhere (mode = None),
    leave the row's genre untouched -- it will be caught by Step 5.
    """
    canonical = movie_mode_genres.get(row["movie_name"])
    if canonical is not None:
        return canonical
    return row["genres"]   # still [] if movie is completely genre-less

df["genres"] = df.apply(apply_movie_mode_genre, axis=1)

n_filled = before_empty_genres - df["genres"].apply(len).eq(0).sum()
print(f"[Genre] Empty-genre rows filled via mode propagation: {n_filled:,}")

# Movies with zero genre data across ALL their rows get the fallback label.
still_empty_mask = df["genres"].apply(len).eq(0)
n_orphaned = still_empty_mask.sum()

if n_orphaned > 0:
    # Use apply to assign a list object per cell; direct loc assignment unpacks the
    # outer list and stores the bare string "Unknown genre" instead of ["Unknown genre"].
    df.loc[still_empty_mask, "genres"] = df.loc[still_empty_mask, "genres"].apply(
        lambda _: list(UNKNOWN_GENRE_FALLBACK)
    )
    print(f"\n[Impute] {n_orphaned:,} rows had no genre data anywhere -> "
          f"imputed as {UNKNOWN_GENRE_FALLBACK}.")
else:
    print(f"\n[Impute] No fully orphaned rows found -- all genres resolved.")

after_empty_genres  = df["genres"].apply(len).eq(0).sum()
after_unknown_movie = df["is_unknown_movie_name"].sum()   # flagged, not removed
print(f"[After]  Empty genres [] rows remaining : {after_empty_genres:,}")
print(f"[After]  'Unknown' movie_name rows (flagged, not dropped): "
      f"{after_unknown_movie:,}")

# Identical review text is an artifact of data construction (emotion-tag explosion).
# Keep first occurrence only; investigation script confirms this is safe.
rows_before_dedup  = len(df)
dup_reviews_before = df["Reviews"].duplicated(keep=False).sum()
print(f"\n[Dedup] Rows sharing identical Reviews text (all copies): "
      f"{dup_reviews_before:,}")

df = df.drop_duplicates(subset=["Reviews"], keep="first")

rows_after_dedup  = len(df)
dup_reviews_after = df["Reviews"].duplicated(keep=False).sum()   # should be 0

print(f"[Dedup] Rows removed     : {rows_before_dedup - rows_after_dedup:,}")
print(f"[Dedup] Rows remaining   : {rows_after_dedup:,}")
print(f"[Dedup] Duplicate Reviews remaining: {dup_reviews_after:,}")


COLORS_BEFORE = "#E07B54"   # warm orange  -> "before" bars
COLORS_AFTER  = "#4C9BE8"   # cool blue    -> "after"  bars
BAR_WIDTH     = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Step 1 -- Data Quality: Before vs. After Cleaning",
             fontsize=14, fontweight="bold", y=1.01)

labels1        = ["'Unknown'\nmovie_name", "Empty\ngenres []"]
before_counts1 = [before_unknown_movie, before_empty_genres]
after_counts1  = [after_unknown_movie,  after_empty_genres]

x1 = range(len(labels1))
bars_b1 = ax1.bar([i - BAR_WIDTH / 2 for i in x1],
                  before_counts1, BAR_WIDTH,
                  label="Before", color=COLORS_BEFORE, edgecolor="white")
bars_a1 = ax1.bar([i + BAR_WIDTH / 2 for i in x1],
                  after_counts1, BAR_WIDTH,
                  label="After",  color=COLORS_AFTER,  edgecolor="white")

ax1.set_xticks(list(x1))
ax1.set_xticklabels(labels1, fontsize=11)
ax1.set_ylabel("Row count", fontsize=11)
ax1.set_title("Missing / Placeholder Data", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.spines[["top", "right"]].set_visible(False)

max1 = max(before_counts1) if max(before_counts1) > 0 else 1
for bar in bars_b1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + max1 * 0.02,
             f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")
for bar in bars_a1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + max1 * 0.02,
             f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")

labels2        = ["Rows with\nDuplicate Reviews"]
before_counts2 = [before_dup_reviews]
after_counts2  = [dup_reviews_after]

x2 = range(len(labels2))
bars_b2 = ax2.bar([i - BAR_WIDTH / 2 for i in x2],
                  before_counts2, BAR_WIDTH,
                  label="Before", color=COLORS_BEFORE, edgecolor="white")
bars_a2 = ax2.bar([i + BAR_WIDTH / 2 for i in x2],
                  after_counts2, BAR_WIDTH,
                  label="After",  color=COLORS_AFTER,  edgecolor="white")

ax2.set_xticks(list(x2))
ax2.set_xticklabels(labels2, fontsize=11)
ax2.set_ylabel("Row count", fontsize=11)
ax2.set_title("Duplicate Reviews (identical text)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.spines[["top", "right"]].set_visible(False)

max2 = max(before_counts2) if max(before_counts2) > 0 else 1
for bar in bars_b2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + max2 * 0.02,
             f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")
for bar in bars_a2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + max2 * 0.02,
             f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")

plt.tight_layout()
plt.savefig(CHART_FILE, dpi=150, bbox_inches="tight")
plt.close()
print(f"\n[Chart] Saved -> {CHART_FILE}")

# CSV cannot store Python lists natively — convert back to string representation.
df["genres"] = df["genres"].apply(lambda g: str(g))

df.to_csv(OUTPUT_FILE, index=False)

print(f"\n[Export] Saved -> {OUTPUT_FILE}  ({len(df):,} rows)")
print("\n[Done]  Step 1 complete.\n")


STEP 1: Missing / Placeholder Data Handling + Deduplication

[Load]  Rows: 46,173  |  Columns: ['Unnamed: 0', 'Ratings', 'Reviews', 'movie_name', 'Resenhas', 'genres', 'Description', 'emotion']

[Parse] `genres` column converted from strings to Python lists.

[Before] 'Unknown' movie_name rows : 40
[Before] Empty genres [] rows       : 880
[Before] Total rows                 : 46,173
[Before] Rows with duplicate Reviews: 33,805

[Flag]  40 rows flagged as 'Unknown' movie_name (column: is_unknown_movie_name).

[Genre] Movies with conflicting genre lists (before fix): 404
[Genre] Empty-genre rows filled via mode propagation: 792

[Impute] 88 rows had no genre data anywhere -> imputed as ['Unknown genre'].
[After]  Empty genres [] rows remaining : 0
[After]  'Unknown' movie_name rows (flagged, not dropped): 40

[Dedup] Rows sharing identical Reviews text (all copies): 33,805
[Dedup] Rows removed     : 26,857
[Dedup] Rows remaining   : 19,316
[Dedup] Duplicate Reviews remaining: 0

[Chart]